# Semantic Search on a Toy Corpus
This notebook reuses the same synthetic corpus as the webapp. We will embed the docs, score a few queries, and compare semantic search with a naive keyword baseline.

In [ ]:
!pip install sentence-transformers pandas matplotlib -q

In [ ]:
from pathlib import Path
import urllib.request
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Works three ways: full repo clone (reads the local file), a standalone
# download of just this notebook, or opened straight from the Colab badge
# (both of the latter fetch the same corpus from GitHub).
RAW_CORPUS_URL = 'https://raw.githubusercontent.com/akashsd/transformers_dbi_staff/main/data/corpus.txt'
corpus_path = Path('..') / 'data' / 'corpus.txt'
if corpus_path.exists():
    docs = corpus_path.read_text(encoding='utf-8').splitlines()
else:
    with urllib.request.urlopen(RAW_CORPUS_URL) as response:
        docs = response.read().decode('utf-8').splitlines()

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
doc_vectors = model.encode(docs, show_progress_bar=True)
len(docs), doc_vectors.shape

In [ ]:
def semantic_search(query, top_k=5):
    query_vector = model.encode([query])
    scores = cosine_similarity(query_vector, doc_vectors)[0]
    ranked = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)[:top_k]
    return [(docs[index], float(score)) for index, score in ranked]

In [ ]:
def keyword_search(query, top_k=5):
    terms = [part for part in query.lower().split() if part]
    results = []
    for doc in docs:
        lowered = doc.lower()
        score = sum(term in lowered for term in terms) / max(len(terms), 1)
        results.append((doc, score))
    return sorted(results, key=lambda item: item[1], reverse=True)[:top_k]

In [ ]:
queries = [
    'fans fighting near the entrance',
    'person overheated in the crowd',
    'someone grabbed my phone in the crowd'
]
for query in queries:
    print('QUERY:', query)
    print('SEMANTIC:', semantic_search(query, 3))
    print('KEYWORD:', keyword_search(query, 3))
    print()

## Exercise
Write a query where semantic beats keyword by a wide margin. Write one where keyword beats semantic.
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE
import matplotlib.pyplot as plt
scores = cosine_similarity(model.encode([q for q in queries]), doc_vectors)
plt.hist(scores.flatten(), bins=20)
plt.title('Cosine similarity distribution')
plt.show()

## What to try next
Open Notebook 3 to search your own CSV data.